In [1]:
import re
import os
import glob
import math
from datetime import datetime
import pandas as pd
from datetime import datetime
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font

In [2]:
print("Current directory:", os.getcwd())

Current directory: /Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/0000_EfficiencyWithIT/SmartSeminary/jupyter_notebooks


# Create DropDown Menues from a separate File

In [3]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter, quote_sheetname
from openpyxl.worksheet.datavalidation import DataValidation

# ------------------------------------------------------------
# FILES
# ------------------------------------------------------------
DropDown_RETRIEVAL = "/Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/0000_EfficiencyWithIT/SmartSeminary/0_source_files/0000_BuildingFiles/source_DropDownMenus_v20260505.xlsx"
target_file = "/Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/0000_EfficiencyWithIT/SmartSeminary/0_source_files/default_DonSheet/DonSheet_default_empty_v20260505.xlsx"

dd_sheet_name = "DropDownMenu"
APPLY_TO_ROW = 5000

# ------------------------------------------------------------
# STEP 1: Read dropdown source workbook into dict (fast + clean)
# ------------------------------------------------------------
df_dd = pd.read_excel(DropDown_RETRIEVAL, dtype=str)

dropdown_lists = {
    col: df_dd[col].dropna().astype(str).str.strip().unique().tolist()
    for col in df_dd.columns
}

# ------------------------------------------------------------
# STEP 2: Load target workbook
# ------------------------------------------------------------
wb = load_workbook(target_file)

# ------------------------------------------------------------
# STEP 3: Create/refresh DropDownMenu sheet inside target workbook
# ------------------------------------------------------------
if dd_sheet_name in wb.sheetnames:
    ws_dd = wb[dd_sheet_name]
    wb.remove(ws_dd)  # remove entirely (faster + avoids leftover values)
ws_dd = wb.create_sheet(dd_sheet_name)

dropdown_ranges = {}
columns_with_false_default = set()

col_idx = 1
for header, values in dropdown_lists.items():
    if not values:
        continue

    ws_dd.cell(row=1, column=col_idx, value=header)

    # write values
    for row_idx, val in enumerate(values, start=2):
        ws_dd.cell(row=row_idx, column=col_idx, value=val)

    last_row = 1 + len(values)
    col_letter = get_column_letter(col_idx)

    dropdown_ranges[header] = f"{quote_sheetname(dd_sheet_name)}!${col_letter}$2:${col_letter}${last_row}"

    if "False" in values:
        columns_with_false_default.add(header)

    col_idx += 1

ws_dd.sheet_state = "hidden"

# ------------------------------------------------------------
# STEP 4: Apply ONLY dropdown validations to all sheets
# ------------------------------------------------------------
for ws in wb.worksheets:
    if ws.title == dd_sheet_name:
        continue

    # Map header -> column index for row 1 (fast lookups)
    headers = [cell.value for cell in ws[1]]
    header_to_idx = {h: i + 1 for i, h in enumerate(headers) if h}

    # Add data validations for headers that exist on the sheet
    for header, formula_range in dropdown_ranges.items():
        col_i = header_to_idx.get(header)
        if not col_i:
            continue

        col_letter = get_column_letter(col_i)

        dv = DataValidation(type="list", formula1=f"={formula_range}", allow_blank=True)
        ws.add_data_validation(dv)
        dv.add(f"${col_letter}$2:${col_letter}${APPLY_TO_ROW}")

        # Optional: set default "False" for blank cells for those dropdowns
        if header in columns_with_false_default:
            max_row = min(ws.max_row, APPLY_TO_ROW)
            for r in range(2, max_row + 1):
                cell = ws[f"{col_letter}{r}"]
                if cell.value is None or str(cell.value).strip() == "":
                    cell.value = "False"

# ------------------------------------------------------------
# STEP 5: Save
# ------------------------------------------------------------
wb.save(target_file)
wb.close()

print("✅ Applied dropdown menus only (no formatting changes).")


✅ Applied dropdown menus only (no formatting changes).


/opt/anaconda3/lib/python3.13/site-packages/openpyxl/reader/excel.py:237: UserWarning: Data Validation extension is not supported and will be removed
  ws_parser.bind_all()


# Creating AutoFreeze, Autofilter

In [4]:
import warnings
from openpyxl import load_workbook
from openpyxl.styles import Alignment
from openpyxl.utils import get_column_letter

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

# ------------------------------------------------------------
# Input / Output
# ------------------------------------------------------------
input_file  = "/Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/0000_EfficiencyWithIT/SmartSeminary/0_source_files/default_DonSheet/DonSheet_default_empty_v20260505.xlsx"
output_file = input_file

# ------------------------------------------------------------
# Load workbook
# ------------------------------------------------------------
wb = load_workbook(input_file)

def format_sheet(ws):
    # ------------------------------------------------------------
    # 1. Header formatting: vertical, bottom-center
    # ------------------------------------------------------------
    header_alignment = Alignment(
        textRotation=90,
        vertical="bottom",
        horizontal="center",
        wrap_text=False
    )

    for cell in ws[1]:
        cell.alignment = header_alignment

    # ------------------------------------------------------------
    # 2. Apply autofilter
    # ------------------------------------------------------------
    ws.auto_filter.ref = ws.dimensions

    # ------------------------------------------------------------
    # 3. Autofit column width (IGNORE header row)
    # ------------------------------------------------------------
    for col_idx in range(1, ws.max_column + 1):
        col_letter = get_column_letter(col_idx)
        max_length = 0

        for row in ws.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
            val = row[0].value
            if val is None:
                continue

            text = str(val)

            # If there are line breaks, use the longest line
            if "\n" in text:
                text = max(text.split("\n"), key=len)

            max_length = max(max_length, len(text))

        ws.column_dimensions[col_letter].width = max(10, min(max_length + 2, 60))

# ------------------------------------------------------------
# Apply formatting to ALL sheets (skip hidden dropdown sheets if desired)
# ------------------------------------------------------------
SKIP_SHEETS = {"DropDown", "DropDownMenu"}  # adjust as needed

for ws in wb.worksheets:
    if ws.title in SKIP_SHEETS:
        continue
    format_sheet(ws)

# ------------------------------------------------------------
# Save to output
# ------------------------------------------------------------
wb.save(output_file)
wb.close()

print(f"✅ Saved formatted workbook to: {output_file}")


✅ Saved formatted workbook to: /Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/0000_EfficiencyWithIT/SmartSeminary/0_source_files/default_DonSheet/DonSheet_default_empty_v20260505.xlsx


# Adding Formulas to Sheets

In [ ]:
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter

def apply_header_based_formulas(
    in_path: str,
    out_path: str,
    rules: dict,
    lookup_col_letter: str = "A",   # the column holding the key (e.g., A has A3, A4...)
    header_row: int = 1,
    data_start_row: int = 2,
    exclude_sheets=("AutoFill_names",),
):
    """
    rules example:
      {
        "Instr Email": '=LET(v,XLOOKUP({key},AutoFill_names!B:B,AutoFill_names!D:D,""),IF(v=0,"",v))',
        "Instr ID":    '=LET(v,XLOOKUP({key},AutoFill_names!B:B,AutoFill_names!E:E,""),IF(v=0,"",v))',
      }
    Use {key} placeholder; it will be replaced with e.g. $A2, $A3...
    """
    wb = load_workbook(in_path)

    for ws in wb.worksheets:
        if ws.title in exclude_sheets:
            continue

        # find last row with any data (simple heuristic)
        last_row = ws.max_row

        # build header -> column index mapping
        header_to_col = {}
        for c in range(1, ws.max_column + 1):
            val = ws.cell(row=header_row, column=c).value
            if isinstance(val, str) and val.strip():
                header_to_col[val.strip()] = c

        for header_name, formula_template in rules.items():
            if header_name not in header_to_col:
                continue

            target_col_idx = header_to_col[header_name]
            target_col_letter = get_column_letter(target_col_idx)

            # fill formula down the column
            for r in range(data_start_row, last_row + 1):
                key_ref = f"${lookup_col_letter}{r}"  # absolute col, relative row
                formula = formula_template.format(key=key_ref)

                ws[f"{target_col_letter}{r}"].value = formula

    wb.save(out_path)


# -------------------------------
# Example usage
# -------------------------------
input_file  = "input.xlsx"
output_file = "output_with_formulas.xlsx"

rules = {
    # If a sheet has header "X", apply formula "Z" to that entire column
    "SomeHeaderX": '=LET(v,XLOOKUP({key},AutoFill_names!B:B,AutoFill_names!D:D,""),IF(v=0,"",v))',
    # Add more headers -> formulas here
}

apply_header_based_formulas(
    in_path=input_file,
    out_path=output_file,
    rules=rules,
    lookup_col_letter="A",  # this is where A3 comes from in your example
)


# ReRunning Formatting after Step 2

In [5]:
import os
import re
import glob
import numpy as np
from copy import copy
from collections import defaultdict
import pandas as pd
import openpyxl
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter, column_index_from_string, quote_sheetname
from openpyxl.worksheet.datavalidation import DataValidation

In [6]:
import os
import glob

from openpyxl import load_workbook
from openpyxl.utils import get_column_letter, column_index_from_string, quote_sheetname
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.worksheet.datavalidation import DataValidation

# ===========================================
# CELL 2: Load workbook + formatting + grouping + autofit
#       + Re-apply dropdown validations (dropdown arrows/UI)
#       + Force most DATA cells to be stored as text (strings)
#       + Apply AutoFilter to ALL columns in ALL processed sheets
#       + Inject InstructorMap formulas for Email/ID based on Instructor Name
# ===========================================

# ------------------------------------------------------------
# 0) Find the most recent XLSX file in the live folder
# ------------------------------------------------------------
DEFAULT_DonSheet = "/Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/0000_EfficiencyWithIT/SmartSeminary/0_source_files/default_DonSheet"
files = glob.glob(os.path.join(DEFAULT_DonSheet, "*.xlsx"))

if not files:
    raise FileNotFoundError(f"No .xlsx files found in {DEFAULT_DonSheet}/")

INPUTFILE = max(files, key=os.path.getmtime)
print(f"Using live file: {INPUTFILE}")

# Load workbook (keep formulas)
wb = load_workbook(INPUTFILE, data_only=False)

# ------------------------------------------------------------
# 1) Rebuild dropdown ranges from hidden DropDownMenu sheet
# ------------------------------------------------------------
def build_dropdown_ranges_from_dd_sheet(wb, dd_sheet_name="DropDownMenu"):
    if dd_sheet_name not in wb.sheetnames:
        return {}, {}

    ws_dd = wb[dd_sheet_name]
    dropdown_ranges = {}
    dropdown_values = {}

    for col_idx in range(1, ws_dd.max_column + 1):
        header = ws_dd.cell(row=1, column=col_idx).value
        if header is None or str(header).strip() == "":
            continue

        values = []
        for r in range(2, ws_dd.max_row + 1):
            v = ws_dd.cell(row=r, column=col_idx).value
            if v is None or str(v).strip() == "":
                continue
            values.append(str(v))

        if not values:
            continue

        col_letter = get_column_letter(col_idx)
        last_row = 1 + len(values)

        dropdown_ranges[header] = f"{quote_sheetname(dd_sheet_name)}!${col_letter}$2:${col_letter}${last_row}"
        dropdown_values[header] = values

    return dropdown_ranges, dropdown_values


def reapply_dropdown_validations(sheet, dropdown_ranges, dropdown_values, max_rows=5000):
    headers = [cell.value for cell in sheet[1]]

    for header, formula_range in dropdown_ranges.items():
        if header not in headers:
            continue

        cidx = headers.index(header) + 1
        cL = get_column_letter(cidx)

        dv = DataValidation(type="list", formula1=f"={formula_range}", allow_blank=True)
        dv.showDropDown = False  # False = show dropdown arrow
        sheet.add_data_validation(dv)
        dv.add(f"${cL}$2:${cL}${max_rows}")

        # Optional: default blanks to "False" if False is an allowed option
        values_list = dropdown_values.get(header, [])
        if "False" in values_list:
            for r in range(2, min(sheet.max_row or 1, max_rows) + 1):
                cell = sheet[f"{cL}{r}"]
                if cell.value is None or str(cell.value).strip() == "":
                    cell.value = "False"


dropdown_ranges, dropdown_values = build_dropdown_ranges_from_dd_sheet(wb, dd_sheet_name="DropDownMenu")

# ------------------------------------------------------------
# 2) Grouping
# ------------------------------------------------------------
GROUP_DEFS = [
    ("B", "K"),
    ("P", "W"),
    ("Y", "AB"),
    ("AD", "AS"),
    ("AX", "BB"),
    ("BD", "BG"),
    ("BI", "BX"),
    ("CC", "CG"),
    ("CI", "CL"),
    ("CN", "DC"),
]


def group_columns(sheet, group_defs=GROUP_DEFS, outline_level=1):
    sheet.sheet_properties.outlinePr.summaryRight = False
    sheet.sheet_properties.outlinePr.summaryBelow = True
    sheet.sheet_properties.outlinePr.applyStyles = True

    max_col = sheet.max_column or 1
    for start_col, end_col in group_defs:
        start_idx = column_index_from_string(start_col)
        end_idx = column_index_from_string(end_col)

        if start_idx > max_col:
            continue

        end_idx = min(end_idx, max_col)
        for col_idx in range(start_idx, end_idx + 1):
            col_letter = get_column_letter(col_idx)
            sheet.column_dimensions[col_letter].outline_level = outline_level

# ------------------------------------------------------------
# 3) Inject InstructorMap formulas (Email & ID derived from Instructor Name)
# IMPORTANT: must happen BEFORE force_all_columns_to_text()
# ------------------------------------------------------------
INSTR_NAME_HDR  = "Instructor Name {Instr Name}"
INSTR_EMAIL_HDR = "Instructor Email {Instr Email}"
INSTR_ID_HDR    = "Instructor ID {Instr ID}"

FORMULA_SHEETS = [
    "CHIS", "DSLE", "GSEM", "MSSN", "NTST", "OTST",
    "PATH", "THST", "MDivHISP_MAPmENGL_MAPmHISP", "MA_Religion", "DMIN",
]

def apply_instructor_map_formulas(wb, sheet_names, max_rows=5000):
    if "InstructorMap" not in wb.sheetnames:
        return  # nothing to do

    for sname in sheet_names:
        if sname not in wb.sheetnames:
            continue

        sheet = wb[sname]
        headers = [cell.value for cell in sheet[1]]

        if INSTR_NAME_HDR not in headers:
            continue

        name_col = headers.index(INSTR_NAME_HDR) + 1
        name_L = get_column_letter(name_col)

        max_r = min(sheet.max_row or 1, max_rows)

        # Email formula
        if INSTR_EMAIL_HDR in headers:
            email_col = headers.index(INSTR_EMAIL_HDR) + 1
            email_L = get_column_letter(email_col)
            for r in range(2, max_r + 1):
                sheet[f"{email_L}{r}"].value = (
                    f'=IF(${name_L}{r}="","",'
                    f'IFERROR(INDEX(InstructorMap!$B:$B, MATCH(${name_L}{r}, InstructorMap!$A:$A, 0)),""))'
                )

        # ID formula
        if INSTR_ID_HDR in headers:
            id_col = headers.index(INSTR_ID_HDR) + 1
            id_L = get_column_letter(id_col)
            for r in range(2, max_r + 1):
                sheet[f"{id_L}{r}"].value = (
                    f'=IF(${name_L}{r}="","",'
                    f'IFERROR(INDEX(InstructorMap!$C:$C, MATCH(${name_L}{r}, InstructorMap!$A:$A, 0)),""))'
                )

# Apply formulas
apply_instructor_map_formulas(wb, FORMULA_SHEETS, max_rows=5000)

# ------------------------------------------------------------
# 4) Force all data cells (rows 2+) to Text (strings) EXCEPT certain columns
# ------------------------------------------------------------
SKIP_TEXT_COLUMNS = {
    "Instructor Name {Instr Name}",
    "Instructor Email {Instr Email}",
    "Instructor ID {Instr ID}",
}

def force_all_columns_to_text(sheet, skip_headers=None, max_rows=5000):
    if skip_headers is None:
        skip_headers = set()

    headers = [cell.value for cell in sheet[1]]

    skip_col_indexes = {
        headers.index(h) + 1
        for h in skip_headers
        if h in headers
    }

    max_row = min(sheet.max_row or 1, max_rows)

    for row in sheet.iter_rows(min_row=2, max_row=max_row, min_col=1, max_col=sheet.max_column):
        for cell in row:
            if cell.value is None:
                continue
            if cell.column in skip_col_indexes:
                continue
            cell.value = str(cell.value)
            cell.number_format = "@"

# ------------------------------------------------------------
# 5) Autofit widths (NO cached values)
# ------------------------------------------------------------
def autofit_columns_ignore_header(sheet, min_width=4, max_width=60, padding=2, max_rows=5000):
    max_row = min(sheet.max_row or 1, max_rows)
    max_col = sheet.max_column or 1

    for col_idx in range(1, max_col + 1):
        max_length = 0
        for r in range(2, max_row + 1):
            v = sheet.cell(row=r, column=col_idx).value
            if v is None:
                continue
            max_length = max(max_length, len(str(v)))

        col_letter = get_column_letter(col_idx)
        sheet.column_dimensions[col_letter].width = max(min_width, min(max_length + padding, max_width))

# ------------------------------------------------------------
# 6) Fixed widths (override AFTER autofit)
# ------------------------------------------------------------
FIXED_WIDTHS = {
    "Notes {not in Banner}": 40,
    "Instructor Name {Instr Name}": 20,
    "Instructor Email {Instr Email}": 20,
    "Instructor ID {Instr ID}": 10,
}

def apply_fixed_widths(sheet, fixed_widths=FIXED_WIDTHS):
    headers = [cell.value for cell in sheet[1]]
    for header, width in fixed_widths.items():
        if header in headers:
            col_idx = headers.index(header) + 1
            sheet.column_dimensions[get_column_letter(col_idx)].width = width

# ------------------------------------------------------------
# 7) Formatting
# ------------------------------------------------------------
thin_grey_border = Border(
    left=Side(style="thin", color="D3D3D3"),
    right=Side(style="thin", color="D3D3D3"),
    top=Side(style="thin", color="D3D3D3"),
    bottom=Side(style="thin", color="D3D3D3"),
)

def format_sheet(sheet):
    headers = [cell.value for cell in sheet[1]]

    # Freeze panes
    if "Notes {not in Banner}" in headers:
        notes_idx = headers.index("Notes {not in Banner}") + 1
        sheet.freeze_panes = f"{get_column_letter(notes_idx + 1)}2"
    else:
        sheet.freeze_panes = "A2"

    # AutoFilter (force full-width, full-height)
    sheet.auto_filter.ref = None
    last_col = get_column_letter(sheet.max_column or 1)
    last_row = sheet.max_row or 1
    sheet.auto_filter.ref = f"A1:{last_col}{last_row}"

    # Header style
    for cell in sheet[1]:
        if cell.value is None or str(cell.value).strip() == "":
            continue
        cell.font = Font(
            name="Arial",
            size=10,
            bold=True,
            color="FF0000" if cell.value == "Notes {not in Banner}" else "000000",
        )
        cell.fill = PatternFill(start_color="D3D3D3", end_color="D3D3D3", fill_type="solid")
        cell.alignment = Alignment(horizontal="center", vertical="bottom", wrap_text=True, text_rotation=90)

    sheet.row_dimensions[1].height = 150

    # Light-blue header block
    start_h = "Don {not in Banner}"
    end_h = "SEM Department {Scacrse Dept}"
    if start_h in headers and end_h in headers:
        start_idx = headers.index(start_h) + 1
        end_idx = headers.index(end_h) + 1
        light_blue_fill = PatternFill(start_color="DDEBF7", end_color="DDEBF7", fill_type="solid")
        for col_idx in range(start_idx, end_idx + 1):
            sheet[f"{get_column_letter(col_idx)}1"].fill = light_blue_fill

    # Grey-out inactive rows (Status == "I")
    if "Status" in headers:
        status_col_idx = headers.index("Status") + 1
        inactive_fill = PatternFill(start_color="D9D9D9", end_color="D9D9D9", fill_type="solid")
        for r in range(2, (sheet.max_row or 1) + 1):
            v = sheet.cell(row=r, column=status_col_idx).value
            if str(v).strip().upper() == "I":
                for c in range(1, (sheet.max_column or 1) + 1):
                    sheet.cell(row=r, column=c).fill = inactive_fill

    # Borders
    for row in sheet.iter_rows(min_row=2, max_row=sheet.max_row or 1, min_col=1, max_col=sheet.max_column or 1):
        for cell in row:
            cell.border = thin_grey_border

    # Fonts: global default Arial 10
    for row in sheet.iter_rows(min_row=1, max_row=sheet.max_row or 1, min_col=1, max_col=sheet.max_column or 1):
        for cell in row:
            f = cell.font or Font()
            cell.font = Font(
                name="Arial",
                size=10,
                bold=f.bold,
                italic=f.italic,
                underline=f.underline,
                color=f.color,
            )

    # Override columns A–G to Arial 12 (DATA ROWS ONLY)
    for row in sheet.iter_rows(min_row=2, max_row=sheet.max_row or 1, min_col=1, max_col=min(7, sheet.max_column or 1)):
        for cell in row:
            f = cell.font or Font()
            cell.font = Font(
                name="Arial",
                size=12,
                bold=f.bold,
                italic=f.italic,
                underline=f.underline,
                color=f.color,
            )

# ------------------------------------------------------------
# 8) Process sheets (dynamic list excluding helpers)
# ------------------------------------------------------------
EXCLUDE_SHEETS = {"DropDownMenu", "InstructorMap"}
export_sheets = [name for name in wb.sheetnames if name not in EXCLUDE_SHEETS]

for name in export_sheets:
    ws = wb[name]

    format_sheet(ws)

    # Force text BEFORE autofit (width uses string length)
    force_all_columns_to_text(ws, skip_headers=SKIP_TEXT_COLUMNS, max_rows=5000)

    group_columns(ws)
    autofit_columns_ignore_header(ws)

    apply_fixed_widths(ws)
    reapply_dropdown_validations(ws, dropdown_ranges, dropdown_values, max_rows=5000)

# Save in place (overwrite live file)
wb.save(INPUTFILE)
wb.close()

print("✅ Live workbook updated in place (with InstructorMap formulas).")


Using live file: /Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/0000_EfficiencyWithIT/SmartSeminary/0_source_files/default_DonSheet/DonSheet_default_empty_v20260505.xlsx
✅ Live workbook updated in place (with InstructorMap formulas).


# Applying Formatting to the DropDownMenu

In [ ]:
import os
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Border, Side, Alignment
from openpyxl.utils import get_column_letter, column_index_from_string
from openpyxl.utils.cell import quote_sheetname
from openpyxl.worksheet.datavalidation import DataValidation

INPUTFILE = "/Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/0000_EfficiencyWithIT/SmartSeminary/0_source_files/0000_BuildingFiles/source_DropDownMenus_v20260405.xlsx"
wb = load_workbook(INPUTFILE)

# ------------------------------------------------------------
# 1) Rebuild dropdown ranges from hidden DropDownMenu sheet
# ------------------------------------------------------------
def build_dropdown_ranges_from_dd_sheet(wb, dd_sheet_name="DropDownMenu"):
    if dd_sheet_name not in wb.sheetnames:
        return {}, {}

    ws_dd = wb[dd_sheet_name]
    dropdown_ranges = {}
    dropdown_values = {}

    for col_idx in range(1, ws_dd.max_column + 1):
        header = ws_dd.cell(row=1, column=col_idx).value
        if header is None or str(header).strip() == "":
            continue

        values = []
        for r in range(2, ws_dd.max_row + 1):
            v = ws_dd.cell(row=r, column=col_idx).value
            if v is None or str(v).strip() == "":
                continue
            values.append(str(v))

        if not values:
            continue

        col_letter = get_column_letter(col_idx)
        last_row = 1 + len(values)

        dropdown_ranges[header] = f"{quote_sheetname(dd_sheet_name)}!${col_letter}$2:${col_letter}${last_row}"
        dropdown_values[header] = values

    return dropdown_ranges, dropdown_values


def clear_data_validations(sheet):
    sheet.data_validations.dataValidation = []


def reapply_dropdown_validations(sheet, dropdown_ranges, dropdown_values, max_rows=5000):
    headers = [cell.value for cell in sheet[1]]

    for header, formula_range in dropdown_ranges.items():
        if header not in headers:
            continue

        cidx = headers.index(header) + 1
        cL = get_column_letter(cidx)

        dv = DataValidation(type="list", formula1=f"={formula_range}", allow_blank=True)
        dv.showDropDown = False
        sheet.add_data_validation(dv)
        dv.add(f"${cL}$2:${cL}${max_rows}")

        values_list = dropdown_values.get(header, [])
        if "False" in values_list:
            for r in range(2, min(sheet.max_row or 1, max_rows) + 1):
                cell = sheet[f"{cL}{r}"]
                if cell.value is None or str(cell.value).strip() == "":
                    cell.value = "False"


dropdown_ranges, dropdown_values = build_dropdown_ranges_from_dd_sheet(wb, dd_sheet_name="DropDownMenu")

# ------------------------------------------------------------
# 2) Grouping
# ------------------------------------------------------------
GROUP_DEFS = [
    ("R", "AB"),
    ("AD", "AW"),
    ("AY", "BJ"),
    ("BK", "BV"),
    ("BX", "CE"),
    ("CG", "DQ")
]

def group_columns(sheet, group_defs=GROUP_DEFS, outline_level=1):
    # Show outline controls and place the summary column on the LEFT
    sheet.sheet_view.showOutlineSymbols = True
    sheet.sheet_properties.outlinePr.summaryRight = False
    sheet.sheet_properties.outlinePr.summaryBelow = True
    sheet.sheet_properties.outlinePr.applyStyles = True

    max_col = sheet.max_column or 1

    for start_col, end_col in group_defs:
        start_idx = column_index_from_string(start_col)
        end_idx = min(column_index_from_string(end_col), max_col)

        if start_idx >= end_idx or start_idx > max_col:
            continue

        # For a left-collapsing outline, Excel needs a summary column on the left.
        # So the first column stays visible, and the columns to its right are grouped.
        detail_start_idx = start_idx + 1

        if detail_start_idx <= end_idx:
            for col_idx in range(detail_start_idx, end_idx + 1):
                col_letter = get_column_letter(col_idx)
                dim = sheet.column_dimensions[col_letter]
                dim.outline_level = outline_level
                dim.hidden = False

            # Mark the left summary column so Excel keeps the collapse handle on the left
            summary_letter = get_column_letter(start_idx)
            sheet.column_dimensions[summary_letter].collapsed = False


# ------------------------------------------------------------
# 3) Force all data cells (rows 2+) to Text (strings)
# ------------------------------------------------------------
def force_all_columns_to_text(sheet, max_rows=5000):
    max_row = min(sheet.max_row or 1, max_rows)

    for row in sheet.iter_rows(min_row=2, max_row=max_row, min_col=1, max_col=sheet.max_column):
        for cell in row:
            if cell.value is None:
                continue
            cell.value = str(cell.value)
            cell.number_format = "@"

# ------------------------------------------------------------
# 4) Autofit widths
# ------------------------------------------------------------
def autofit_columns_ignore_header(sheet, min_width=4, max_width=60, padding=2, max_rows=5000):
    max_row = min(sheet.max_row or 1, max_rows)
    max_col = sheet.max_column or 1

    for col_idx in range(1, max_col + 1):
        max_length = 0
        for r in range(2, max_row + 1):
            v = sheet.cell(row=r, column=col_idx).value
            if v is None:
                continue
            max_length = max(max_length, len(str(v)))

        col_letter = get_column_letter(col_idx)
        sheet.column_dimensions[col_letter].width = max(min_width, min(max_length + padding, max_width))

# ------------------------------------------------------------
# 5) Fixed widths
# ------------------------------------------------------------
FIXED_WIDTHS = {
    "Notes {not in Banner}": 40,
    "Instructor Name {Instr Name}": 20,
    "Program {not in Banner}": 12,
    "Catalog Title": 25,
    "Section Title": 25,
    "Schedule Type {not in Banner}": 15,
    "Instruction Method {Inst Method}": 10,
    "Building {Meet Bldg}": 30,
    "Instructor Email {Instr Email}": 20,
    "Instructor ID {Instr ID}": 10,
    "load/contract {not in Banner}": 15,
    "Reason for Contract {not in Banner}": 15,
    "account to be charged {not in Banner}": 25,
}

def apply_fixed_widths(sheet, fixed_widths=FIXED_WIDTHS):
    headers = [cell.value for cell in sheet[1]]
    for header, width in fixed_widths.items():
        if header in headers:
            col_idx = headers.index(header) + 1
            sheet.column_dimensions[get_column_letter(col_idx)].width = width

# ------------------------------------------------------------
# 6) Formatting
# ------------------------------------------------------------
thin_grey_border = Border(
    left=Side(style="thin", color="D3D3D3"),
    right=Side(style="thin", color="D3D3D3"),
    top=Side(style="thin", color="D3D3D3"),
    bottom=Side(style="thin", color="D3D3D3"),
)

def format_sheet(sheet):
    headers = [cell.value for cell in sheet[1]]

    # Freeze panes
    if "Notes {not in Banner}" in headers:
        notes_idx = headers.index("Notes {not in Banner}") + 1
        sheet.freeze_panes = f"{get_column_letter(notes_idx + 1)}2"
    else:
        sheet.freeze_panes = "A2"

    # AutoFilter
    sheet.auto_filter.ref = None
    last_col = get_column_letter(sheet.max_column or 1)
    last_row = sheet.max_row or 1
    sheet.auto_filter.ref = f"A1:{last_col}{last_row}"

    # Header style
    for cell in sheet[1]:
        if cell.value is None or str(cell.value).strip() == "":
            continue
        cell.font = Font(
            name="Arial",
            size=10,
            bold=True,
            color="FF0000" if cell.value == "Notes {not in Banner}" else "000000",
        )
        cell.fill = PatternFill(start_color="D3D3D3", end_color="D3D3D3", fill_type="solid")
        cell.alignment = Alignment(
            horizontal="center",
            vertical="bottom",
            wrap_text=True,
            text_rotation=90
        )

    sheet.row_dimensions[1].height = 150

    # Light-blue header block
    start_h = "Don {not in Banner}"
    end_h = "SEM Department {Scacrse Dept}"
    if start_h in headers and end_h in headers:
        start_idx = headers.index(start_h) + 1
        end_idx = headers.index(end_h) + 1
        light_blue_fill = PatternFill(start_color="DDEBF7", end_color="DDEBF7", fill_type="solid")
        for col_idx in range(start_idx, end_idx + 1):
            sheet[f"{get_column_letter(col_idx)}1"].fill = light_blue_fill

    # Borders
    for row in sheet.iter_rows(min_row=2, max_row=sheet.max_row or 1, min_col=1, max_col=sheet.max_column or 1):
        for cell in row:
            cell.border = thin_grey_border

    # Fonts: global default Arial 10
    for row in sheet.iter_rows(min_row=1, max_row=sheet.max_row or 1, min_col=1, max_col=sheet.max_column or 1):
        for cell in row:
            f = cell.font or Font()
            cell.font = Font(
                name="Arial",
                size=10,
                bold=f.bold,
                italic=f.italic,
                underline=f.underline,
                color=f.color,
            )

    # Override columns A–G to Arial 12 (DATA ROWS ONLY)
    for row in sheet.iter_rows(min_row=2, max_row=sheet.max_row or 1, min_col=1, max_col=min(7, sheet.max_column or 1)):
        for cell in row:
            f = cell.font or Font()
            cell.font = Font(
                name="Arial",
                size=12,
                bold=f.bold,
                italic=f.italic,
                underline=f.underline,
                color=f.color,
            )
    # Blue header cells for selected columns
    blue_headers = {
        "pre-work # of weeks {not in Banner}",
        "pre-work hours/week {not in Banner}",
        "pre-work hours/period {not in Banner}",
        "intensive # of weeks {not in Banner}",
        "intensive hours/week {not in Banner}",
        "intensive hours/period {not in Banner}",
        "post-work # of weeks {not in Banner}",
        "post-work hours/week {not in Banner}",
        "post-work hours/period {not in Banner}",
        "sum of weeks (for check) {not in Banner}",
        "total contract hours {not in Banner}",
        "Year {not in Banner}",
        "Semester {not in Banner}",
        "Remote Employee {not in Banner}",
        "% Responsibility, load/contract {not in Banner}",
        "Reason for Contract {not in Banner}",
        "costs per credit {not in Banner}",
        "total costs {not in Banner}",
        "dept budget {not in Banner}",
        "account to be charged {not in Banner}",
        "SEM Department {Scacrse Dept}",
    }

    blue_fill = PatternFill(start_color="00B0F0", end_color="00B0F0", fill_type="solid")

    for col_idx, header in enumerate(headers, start=1):
        if header in blue_headers:
            sheet.cell(row=1, column=col_idx).fill = blue_fill


# ------------------------------------------------------------
# 7) Apply ONLY to sheet "DropDownMenu"
# ------------------------------------------------------------
TARGET_SHEET = "DropDownMenu"

if TARGET_SHEET not in wb.sheetnames:
    raise ValueError(f"Sheet '{TARGET_SHEET}' not found in workbook.")

ws = wb[TARGET_SHEET]

format_sheet(ws)
force_all_columns_to_text(ws, max_rows=5000)
group_columns(ws)
autofit_columns_ignore_header(ws)
apply_fixed_widths(ws)
clear_data_validations(ws)
reapply_dropdown_validations(ws, dropdown_ranges, dropdown_values, max_rows=5000)

# ------------------------------------------------------------
# 8) Safe save in place via temp file
# ------------------------------------------------------------
temp_file = INPUTFILE.replace(".xlsx", "_temp.xlsx")
wb.save(temp_file)
wb.close()

os.replace(temp_file, INPUTFILE)

print("✅ Workbook updated in place. Code applied only to sheet 'DropDownMenu'.")

✅ Workbook updated in place. Code applied only to sheet 'DropDownMenu'.


## Apply same formatting to the default DonSheet (modifications were needed)

In [ ]:
import os
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Border, Side, Alignment
from openpyxl.utils import get_column_letter, column_index_from_string
from openpyxl.utils.cell import quote_sheetname
from openpyxl.worksheet.datavalidation import DataValidation

INPUTFILE = "/Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/0000_EfficiencyWithIT/SmartSeminary/0_source_files/default_DonSheet/DonSheet_default_empty_v20260506.xlsx"
wb = load_workbook(INPUTFILE)

# ------------------------------------------------------------
# 1) Rebuild dropdown ranges from hidden DropDownMenu sheet
# ------------------------------------------------------------
def build_dropdown_ranges_from_dd_sheet(wb, dd_sheet_name="DropDownMenu"):
    if dd_sheet_name not in wb.sheetnames:
        return {}, {}

    ws_dd = wb[dd_sheet_name]
    dropdown_ranges = {}
    dropdown_values = {}

    for col_idx in range(1, ws_dd.max_column + 1):
        header = ws_dd.cell(row=1, column=col_idx).value
        if header is None or str(header).strip() == "":
            continue

        values = []
        for r in range(2, ws_dd.max_row + 1):
            v = ws_dd.cell(row=r, column=col_idx).value
            if v is None or str(v).strip() == "":
                continue
            values.append(str(v))

        if not values:
            continue

        col_letter = get_column_letter(col_idx)
        last_row = 1 + len(values)

        dropdown_ranges[header] = f"{quote_sheetname(dd_sheet_name)}!${col_letter}$2:${col_letter}${last_row}"
        dropdown_values[header] = values

    return dropdown_ranges, dropdown_values


def clear_data_validations(sheet):
    sheet.data_validations.dataValidation = []


def reapply_dropdown_validations(sheet, dropdown_ranges, dropdown_values, max_rows=5000):
    headers = [cell.value for cell in sheet[1]]

    for header, formula_range in dropdown_ranges.items():
        if header not in headers:
            continue

        cidx = headers.index(header) + 1
        cL = get_column_letter(cidx)

        dv = DataValidation(type="list", formula1=f"={formula_range}", allow_blank=True)
        dv.showDropDown = False
        sheet.add_data_validation(dv)
        dv.add(f"${cL}$2:${cL}${max_rows}")

        values_list = dropdown_values.get(header, [])
        if "False" in values_list:
            for r in range(2, min(sheet.max_row or 1, max_rows) + 1):
                cell = sheet[f"{cL}{r}"]
                if cell.value is None or str(cell.value).strip() == "":
                    cell.value = "False"


dropdown_ranges, dropdown_values = build_dropdown_ranges_from_dd_sheet(wb, dd_sheet_name="DropDownMenu")

# ------------------------------------------------------------
# 2) Grouping
# ------------------------------------------------------------
GROUP_DEFS = [
    ("Q", "AB"),
    ("AC", "AW"),
    ("AX", "BJ"),
    ("BK", "BV"),
    ("BW", "CE"),
    ("CF", "DQ")
]
def clear_existing_column_grouping(sheet):
    for col_letter, dim in sheet.column_dimensions.items():
        dim.outline_level = 0
        dim.hidden = False
        dim.collapsed = False

def group_columns(sheet, group_defs=GROUP_DEFS, outline_level=1):
    # Remove/override all existing column grouping first
    clear_existing_column_grouping(sheet)

    # Show outline controls and place the summary column on the LEFT
    sheet.sheet_view.showOutlineSymbols = True
    sheet.sheet_properties.outlinePr.summaryRight = False
    sheet.sheet_properties.outlinePr.summaryBelow = True
    sheet.sheet_properties.outlinePr.applyStyles = True

    max_col = sheet.max_column or 1

    for start_col, end_col in group_defs:
        start_idx = column_index_from_string(start_col)
        end_idx = min(column_index_from_string(end_col), max_col)

        if start_idx >= end_idx or start_idx > max_col:
            continue

        detail_start_idx = start_idx + 1

        if detail_start_idx <= end_idx:
            for col_idx in range(detail_start_idx, end_idx + 1):
                col_letter = get_column_letter(col_idx)
                dim = sheet.column_dimensions[col_letter]
                dim.outline_level = outline_level
                dim.hidden = False
                dim.collapsed = False

            summary_letter = get_column_letter(start_idx)
            sheet.column_dimensions[summary_letter].collapsed = False


# ------------------------------------------------------------
# 3) Force all data cells (rows 2+) to Text (strings)
# ------------------------------------------------------------
def force_all_columns_to_text(sheet, max_rows=5000):
    max_row = min(sheet.max_row or 1, max_rows)

    for row in sheet.iter_rows(min_row=2, max_row=max_row, min_col=1, max_col=sheet.max_column):
        for cell in row:
            if cell.value is None:
                continue
            cell.value = str(cell.value)
            cell.number_format = "@"

# ------------------------------------------------------------
# 4) Autofit widths
# ------------------------------------------------------------
def autofit_columns_ignore_header(sheet, min_width=4, max_width=60, padding=2, max_rows=5000):
    max_row = min(sheet.max_row or 1, max_rows)
    max_col = sheet.max_column or 1

    for col_idx in range(1, max_col + 1):
        max_length = 0
        for r in range(2, max_row + 1):
            v = sheet.cell(row=r, column=col_idx).value
            if v is None:
                continue
            max_length = max(max_length, len(str(v)))

        col_letter = get_column_letter(col_idx)
        sheet.column_dimensions[col_letter].width = max(min_width, min(max_length + padding, max_width))

# ------------------------------------------------------------
# 5) Fixed widths
# ------------------------------------------------------------
FIXED_WIDTHS = {
"Notes {not in Banner}": 40,
"Program {not in Banner}": 12,
"Catalog Title": 25,
"Section Title": 25,
"Schedule Type {not in Banner}": 15,
"Instruction Method {Inst Method}": 10,
"Meeting Type": 5,
"Semester Start Date {Soaterm Start Date}": 10,
"Pre-work Start Date {not in Banner}": 10,
"Pre-work End Date {not in Banner}": 10,
"Intensive Period Start Date {Meet Start Date}": 10,
"Intensive Period End Date {Meet End Date}": 10,
"Post-work Start Date {not in Banner}": 10,
"Post-work End Date {not in Banner}": 10,
"Semester End Date {Soaterm End Date}": 10,
"Course Beginning Time {Meet Beg Time}": 5,
"Course Ending Time {Meet End Time}": 5,
"Year {not in Banner}": 5,
"Semester {not in Banner}": 7,
"Room {Meet Room}": 6,
"Building {Meet Bldg}": 30,
"Instructor Name {Instr Name}": 20,
"Instructor Email {Instr Email}": 20,
"Instructor ID {Instr ID}": 10,
"load/contract {not in Banner}": 12,
"Reason for Contract {not in Banner}": 20,
"costs per credit {not in Banner}": 5,
"account to be charged {not in Banner}": 25,
"SEM Department {Scacrse Dept}": 5,
"dean email": 20,
"VP finance email": 20,
"HR email": 20,
}

def apply_fixed_widths(sheet, fixed_widths=FIXED_WIDTHS):
    headers = [cell.value for cell in sheet[1]]
    for header, width in fixed_widths.items():
        if header in headers:
            col_idx = headers.index(header) + 1
            sheet.column_dimensions[get_column_letter(col_idx)].width = width

# ------------------------------------------------------------
# 6) Formatting
# ------------------------------------------------------------
thin_grey_border = Border(
    left=Side(style="thin", color="D3D3D3"),
    right=Side(style="thin", color="D3D3D3"),
    top=Side(style="thin", color="D3D3D3"),
    bottom=Side(style="thin", color="D3D3D3"),
)

def format_sheet(sheet):
    headers = [cell.value for cell in sheet[1]]

    # Freeze panes
    if "Notes {not in Banner}" in headers:
        notes_idx = headers.index("Notes {not in Banner}") + 1
        sheet.freeze_panes = f"{get_column_letter(notes_idx + 1)}2"
    else:
        sheet.freeze_panes = "A2"

    # AutoFilter
    sheet.auto_filter.ref = None
    last_col = get_column_letter(sheet.max_column or 1)
    last_row = sheet.max_row or 1
    sheet.auto_filter.ref = f"A1:{last_col}{last_row}"

    # Header style
    for cell in sheet[1]:
        if cell.value is None or str(cell.value).strip() == "":
            continue
        cell.font = Font(
            name="Arial",
            size=10,
            bold=True,
            color="FF0000" if cell.value == "Notes {not in Banner}" else "000000",
        )
        cell.fill = PatternFill(start_color="D3D3D3", end_color="D3D3D3", fill_type="solid")
        cell.alignment = Alignment(
            horizontal="center",
            vertical="bottom",
            wrap_text=True,
            text_rotation=90
        )

    sheet.row_dimensions[1].height = 150

    # Light-blue header block
    start_h = "Don {not in Banner}"
    end_h = "SEM Department {Scacrse Dept}"
    if start_h in headers and end_h in headers:
        start_idx = headers.index(start_h) + 1
        end_idx = headers.index(end_h) + 1
        light_blue_fill = PatternFill(start_color="DDEBF7", end_color="DDEBF7", fill_type="solid")
        for col_idx in range(start_idx, end_idx + 1):
            sheet[f"{get_column_letter(col_idx)}1"].fill = light_blue_fill

    # Borders
    for row in sheet.iter_rows(min_row=2, max_row=sheet.max_row or 1, min_col=1, max_col=sheet.max_column or 1):
        for cell in row:
            cell.border = thin_grey_border

    # Fonts: global default Arial 10
    for row in sheet.iter_rows(min_row=1, max_row=sheet.max_row or 1, min_col=1, max_col=sheet.max_column or 1):
        for cell in row:
            f = cell.font or Font()
            cell.font = Font(
                name="Arial",
                size=10,
                bold=f.bold,
                italic=f.italic,
                underline=f.underline,
                color=f.color,
            )

    # Override columns A–G to Arial 12 (DATA ROWS ONLY)
    for row in sheet.iter_rows(min_row=2, max_row=sheet.max_row or 1, min_col=1, max_col=min(7, sheet.max_column or 1)):
        for cell in row:
            f = cell.font or Font()
            cell.font = Font(
                name="Arial",
                size=12,
                bold=f.bold,
                italic=f.italic,
                underline=f.underline,
                color=f.color,
            )
    # Blue header cells for selected columns
    blue_headers = {
        "pre-work # of weeks {not in Banner}",
        "pre-work hours/week {not in Banner}",
        "pre-work hours/period {not in Banner}",
        "intensive # of weeks {not in Banner}",
        "intensive hours/week {not in Banner}",
        "intensive hours/period {not in Banner}",
        "post-work # of weeks {not in Banner}",
        "post-work hours/week {not in Banner}",
        "post-work hours/period {not in Banner}",
        "sum of weeks (for check) {not in Banner}",
        "total contract hours {not in Banner}",
        "Year {not in Banner}",
        "Semester {not in Banner}",
        "Remote Employee {not in Banner}",
        "% Responsibility, load/contract {not in Banner}",
        "Reason for Contract {not in Banner}",
        "costs per credit {not in Banner}",
        "total costs {not in Banner}",
        "dept budget {not in Banner}",
        "account to be charged {not in Banner}",
        "SEM Department {Scacrse Dept}",
    }

    blue_fill = PatternFill(start_color="00B0F0", end_color="00B0F0", fill_type="solid")

    for col_idx, header in enumerate(headers, start=1):
        if header in blue_headers:
            sheet.cell(row=1, column=col_idx).fill = blue_fill


# ------------------------------------------------------------
# 7) Apply to ALL sheets (except DropDownMenu)
# ------------------------------------------------------------
for sheet_name in wb.sheetnames:
    if sheet_name == "DropDownMenu":
        continue  # skip the dropdown source sheet

    ws = wb[sheet_name]

    format_sheet(ws)
    force_all_columns_to_text(ws, max_rows=5000)
    group_columns(ws)
    autofit_columns_ignore_header(ws)
    apply_fixed_widths(ws)
    clear_data_validations(ws)
    reapply_dropdown_validations(ws, dropdown_ranges, dropdown_values, max_rows=5000)

# ------------------------------------------------------------
# 8) Safe save in place via temp file
# ------------------------------------------------------------
temp_file = INPUTFILE.replace(".xlsx", "_temp.xlsx")
wb.save(temp_file)
wb.close()

os.replace(temp_file, INPUTFILE)

print("✅ Workbook updated in place. Code applied only to sheet 'DropDownMenu'.")

✅ Workbook updated in place. Code applied only to sheet 'DropDownMenu'.
